# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [7]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [95]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Loan Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [9]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/complaints.csv",
    metadata_columns=[
      "Date received", 
      "Product", 
      "Sub-product", 
      "Issue", 
      "Sub-issue", 
      "Consumer complaint narrative", 
      "Company public response", 
      "Company", 
      "State", 
      "ZIP code", 
      "Tags", 
      "Consumer consent provided?", 
      "Submitted via", 
      "Date sent to company", 
      "Company response to consumer", 
      "Timely response?", 
      "Consumer disputed?", 
      "Complaint ID"
    ]
)

loan_complaint_data = loader.load()

for doc in loan_complaint_data:
    doc.page_content = doc.metadata["Consumer complaint narrative"]

Let's look at an example document to see if everything worked as expected!

In [10]:
loan_complaint_data[0]

Document(metadata={'source': './data/complaints.csv', 'row': 0, 'Date received': '03/27/25', 'Product': 'Student loan', 'Sub-product': 'Federal student loan servicing', 'Issue': 'Dealing with your lender or servicer', 'Sub-issue': 'Trouble with how payments are being handled', 'Consumer complaint narrative': "The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers.", 'Company public response': 'None', 'Company'

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "LoanComplaints".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [11]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    loan_complaint_data,
    embeddings,
    location=":memory:",
    collection_name="LoanComplaints"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [12]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [13]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [14]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [15]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [16]:
naive_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans appears to be problems related to mismanagement and errors by loan servicers. These issues include errors in loan balances, misapplied payments, wrongful denials of payment plans, incorrect or outdated information on credit reports, and difficulties in applying payments towards principal. Many complaints also involve lack of proper communication, unauthorized transfers of loans without proper notification, and disputes over interest accrual and loan amounts.'

In [17]:
naive_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided context, yes, some complaints did not get handled in a timely manner. Specifically, one complaint from a consumer filed on 03/28/25 with company MOHELA was marked as "Timely response?": No, indicating it was not handled in the expected timeframe. Additionally, there are several complaints where consumers reported that their issues remained unresolved for extended periods, such as over a year or nearly 18 months, and they expressed frustration about the lack of resolution despite their repeated efforts.'

In [18]:
naive_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to a combination of factors highlighted in the complaints:\n\n1. **Inadequate or Misleading Information:** Borrowers were often unaware of when their repayment was supposed to start or how interest and balances would accumulate, especially when loans were transferred between servicers without proper notification. Many received no clear instructions or explanations about repayment terms, interest accrual, or due dates.\n\n2. **High Interest and Growing Balances:** Even when payments were made, interest continued to accumulate, sometimes causing balances to grow or remain high despite ongoing payments. Borrowers felt that interest was being applied in a way that made it difficult to pay down principal or reduce debt.\n\n3. **Lack of Communication or Notification:** Several complaints indicated borrowers were not informed of changes in their loan status, transfers between servicers, or delinquency notices. This lack of communication led

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [19]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(loan_complaint_data, )

We'll construct the same chain - only changing the retriever.

In [20]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [21]:
bm25_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans appears to be dealing with the lender or servicer, particularly issues related to miscommunication, incorrect information, or perceived unfair practices, such as fee disputes, trouble with payment application, or receiving inaccurate loan information.'

In [22]:
bm25_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, it appears that multiple complaints were handled in a timely manner, specifically indicated by the response status "Yes" under "Timely response?" for the complaints from rows 509, 288, and 423. All these complaints received responses from the companies within the expected timeframe. \n\nThere is no evidence in the data that any complaints were not handled in a timely manner.'

In [23]:
bm25_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People fail to pay back their loans for various reasons, including issues with their payment plans, miscommunication or lack of communication from the loan servicers, and complications arising from how the loans are managed. Specific reasons identified in the complaints include:\n\n- Being steered into incorrect or unsuitable forbearance options, leading to increased interest and higher principal amounts.\n- Loan transfers to new servicers (like Aidvantage) without proper notification, resulting in confusion and lack of awareness about account status or autopay discontinuation.\n- Failure of servicers to respond to requests for forbearance, deferment, or other repayment arrangements, leading to unpaid bills and negative credit impacts.\n- Errors in payment processing, such as reversed payments or inability to make consistent payments due to administrative mistakes.\n- Lack of proper communication about changes in account status, repayment obligations, or eligibility for discharge or s

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

#### ✅ ANSWER

BM25 is best at answering specific / direct questions. It does so by simply finding the document with key words that match the query at a high frequency.

Example: “Ubuntu 20.04 install NVIDIA driver 470”
Why BM25 wins:
•	Technical queries with version numbers or specific commands match better with exact tokens. There is likely a small number of documents whose keywords match the above and BM25 will prioritize these.
•	Embedding models often treat numbers and commands less precisely. It may retrieve documents for generally how to install different drivers on Ubuntu and certainly not for the specified version.

For factual, high-precision queries, BM25 is often the more reliable choice.

Other examples:
- "Python list comprehension syntax"
- "Section 230 of the Communications Decency Act"
- "ISO 27001 compliance checklist PDF"
- "New York Times January 5 2020 front page headline"

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [24]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [25]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [26]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided information, the most common issue with loans appears to involve errors and mismanagement related to loan balances, payment information, and communication. Specific recurring problems include incorrect or inconsistent loan balances, misapplied payments, lack of clarity about interest and repayment details, and mishandling of personal data. These issues lead to confusion, inaccurate credit reporting, and sometimes violations of privacy laws.'

In [27]:
contextual_compression_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, it appears that at least one complaint did not get handled in a timely manner. Specifically, the complaint about the student loan account review and related issues at Maximus Federal Services, Inc. has been open for nearly 18 months without resolution, despite the consumer requesting responses over the course of more than a year. The complaint indicates significant delays and a lack of resolution over an extended period. \n\nAdditionally, the complaint regarding the original issue not being addressed by EdFinancial Services involved a delay of over 2-3 weeks and ongoing issues, though the response indicates it was closed with explanation.\n\nOverall, yes, some complaints—particularly the case involving Maximus—did not get handled in a timely manner, with delays extending over a year.'

In [28]:
contextual_compression_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for several reasons, including:\n\n1. Lack of awareness or understanding: Some borrowers were not informed by their financial aid officers that they needed to repay their loans. They were unaware of the repayment obligations until they encountered issues later on.\n\n2. Compounding interest and increasing balances: Borrowers often found that interest continued to accrue even when they could not make payments, causing their balances to grow over time despite payments.\n\n3. Difficult financial circumstances: Many borrowers faced financial hardships that made consistent repayment challenging. They struggled with the affordability of increased payments if they tried to pay more or found themselves stuck with accumulating interest if they chose forbearance or deferment.\n\n4. Lack of clear communication and documentation: Issues such as incorrect or confusing account information, failure of lenders or servicers to notify borrowers of due payments, and

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [29]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [30]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [31]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

"Based on the provided complaints data, the most common issue with loans appears to be problems related to mishandling by loan servicers, including errors in loan balances, misapplied payments, wrongful reporting of delinquencies, delays or disruptions in the application or approval of income-driven repayment plans, incorrect account classifications, and poor communication or customer service. Many complaints also involve disputes over loan terms, interest calculations, and improper transfer or reporting of account status, which can adversely impact borrowers' credit and financial stability."

In [32]:
multi_query_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, yes, some complaints did not get handled in a timely manner. Specifically:\n\n- One complaint involving trouble with how payments are being handled at Mohela was marked as "Timely response?" No, indicating it was not handled promptly, as it was over 2-3 weeks without resolution.\n- Another case regarding account status incorrect at Maximus Federal Services was marked as "Timely response?" No, and it was noted that it was more than 10 days with no resolution, even after Promised timeframes.\n- A complaint about incorrect information reported by Nelnet was also marked as "Timely response?" No, with over 30 days passed and complaint still unresolved.\n\nAdditionally, several complaints that involved delays or failed follow-up (e.g., no response after over 30 days, or waiting several weeks with no resolution) confirm that not all complaints were addressed in a timely manner.'

In [33]:
multi_query_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to a combination of factors highlighted in the complaints:\n\n1. **Misleading or Lack of Information**: Borrowers often report being misled by loan servicers about their repayment options, interest accrual, and forgiveness programs. Many were not informed about options like income-driven repayment plans, rehabilitation, or other legal alternatives that could help manage or reduce their debt.\n\n2. **Interest Accumulation and Capitalization**: Several reports mention that interest continued to accrue and compound, especially during forbearance or deferment periods, causing the total debt to balloon significantly beyond the original principal.\n\n3. **Inadequate or Wrongful Handling by Loan Servicers**: Complaints include issues like misapplied payments, errors in loan balances, failure to inform borrowers of their rights, or coercive practices such as forced consolidation instead of enrolling in more favorable repayment plans.\n\n4. *

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

### ✅ ANSWER
Often there are many ways to portray the same piece of information for example:
- The Capital of France is Paris
- The main political center of France is Paris
- The epicenter of France is Paris

Recall measures the likelyhood that a relevant document is retrieved. 
By formulating different versions of the same question, you "target" the vector store from different "angles" increasing the chance that you hit and recall a relevant document no matter how that key piece of information is expressed. 

Also different user may ask the same question in different ways. Through question generation we may be able to reformulate the question in the way that that information is expressed in the vector store.

In class we also talked about pre-pending additional explanatory information to a document to improve recall and relevance. If a document is retrieved from a reformulated question it's possible to add details to the chunk that makes it more relateable to the original query.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [34]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = loan_complaint_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [35]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [36]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [37]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [38]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [39]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issues with loans, particularly student loans, include problems related to incorrect information on credit reports, disputes over loan balances and interest rates, errors in loan balances, misapplied payments, wrongful denials of payment plans, and misconduct by loan servicers such as failure to verify debt legitimacy and illegal credit reporting. \n\nIf we analyze the complaints, some recurring themes are:\n- Errors and inaccuracies in credit reporting.\n- Discrepancies or unjustified changes in interest rates.\n- Mismanagement or errors by loan servicers.\n- Issues related to loan balance and payment application.\n\nTherefore, the most common issue appears to be errors or misconduct by loan servicers affecting loan balances, interest rates, and credit reporting accuracy.'

In [40]:
parent_document_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, the complaints from the dataset indicate that several requests were not handled in a timely manner. Specifically:\n\n- The complaint with ID 12709087 dated 03/28/25 mentions that the company believed it acted appropriately, but the response was "No" in terms of timely response, and the consumer stated they had not heard back despite multiple follow-ups.\n- Similarly, complaint ID 12935889 received a "No" for timeliness, indicating the response was not prompt.\n- Other complaints also describe prolonged wait times, delays, or lack of response, such as waiting on hold for hours or not receiving responses within the expected timeframes.\n\nTherefore, yes, some complaints did not get handled in a timely manner according to the records available.'

In [41]:
parent_document_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including financial hardship, mismanagement by loan servicers, lack of proper communication or notification about payment obligations, and long-term consequences of their educational choices. Some specific factors highlighted in the complaints include:\n\n- Severe financial hardship after graduation, making it difficult to make loan payments.\n- Lack of clear information or notification about when payments were expected to begin.\n- Issues with loan servicing agencies, such as failing to notify borrowers about payment due dates, payment plans, or changes in loan ownership.\n- Relying on deferment or forbearance options that increased the total interest owed.\n- Burdens related to the long-term financial consequences of attending certain schools, especially when misrepresentations about the value of their education and job prospects were involved.\n\nIn summary, inability to pay back loans often stemmed from financial difficult

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [42]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [43]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [44]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided complaints data, appears to be problems related to "Dealing with your lender or servicer." This includes issues such as mismanagement, incorrect reporting of loan status (e.g., showing delinquency when payments are current), errors in loan balances and interest calculation, failure to provide necessary documentation, and poor communication, including unnotified transfer of servicers, unresponsive customer service, and mishandling of repayment or forgiveness applications. \n\nMany complaints highlight that borrowers face significant difficulties due to lack of transparency, inaccurate information, and administrative errors by loan servicers like Maximus, Nelnet, MOHELA, EdFinancial, and Aidvantage.\n\nHence, the most common issue tends to be: **Problems related to handling, reporting, or communication with loan lenders or servicers.**'

In [45]:
ensemble_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the information provided in the complaints data, yes, some complaints indicate that complaints did not get handled in a timely manner. Specifically:\n\n- Complaint ID 12744910 (Row 400), submitted on 03/31/25, was marked as "Timely response?": Yes, but the complaint describes delays and an issue with potential wrongful reporting and unresolved dispute, suggesting the process was still delayed.\n- Complaint ID 12668396 (Row 95), submitted on 04/21/25, was marked as "Timely response?": No, indicating it was not handled in a timely manner.\n- Complaint ID 12654977 (Row 238), submitted on 03/25/25, was marked as "Timely response?": No, and it explicitly states "they failed to keep up to date records" and "long wait times," indicating delays.\n- Other complaints mention delays, long wait times, or responses that were not received or that took more than the expected period.\n\nIn addition, some complaints show that the companies responded late, or records indicate delays in resolut

In [46]:
ensemble_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People often fail to pay back their loans due to multiple interconnected reasons highlighted in the complaints:\n\n1. Lack of Clear and Accurate Information: Borrowers report being misled or inadequately informed about interest accrual, repayment options, loan balances, and legal rights. For example, many were unaware of how interest compounds or that certain repayment plans could help manage debt.\n\n2. Unmanageable Repayment Options: Borrowers face difficulties increasing monthly payments to pay off loans faster without sacrificing essential expenses. Options like forbearance or deferment often lead to ongoing interest accumulation, making the debt worse over time.\n\n3. Servicer Practices and Errors: Complaints cite problematic servicing behaviors, such as misreporting account statuses, failing to notify borrowers of changes or delinquency, or mishandling payment applications, which can lead to missed payments or default.\n\n4. Lack of Transparency and Communication Failures: Many 

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [47]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [48]:
semantic_documents = semantic_chunker.split_documents(loan_complaint_data[:20])

Let's create a new vector store.

In [49]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Loan_Complaint_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [50]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [51]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [52]:
semantic_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints data, the most common issue with loans appears to be problems related to the handling and servicing of federal student loans. Specific frequent issues include:\n\n- Struggling to repay or problems with repayment plans (e.g., miscalculations, incorrect payment amounts, or changes in payment plans).\n- Dealing with loan servicers or lenders—such as lack of transparency, difficulty in obtaining accurate information, or miscommunication.\n- Issues with reporting and credit reporting, including improper reporting of defaults or delinquencies.\n- Problems with loan forgiveness, cancellation, or discharge processes.\n- Unauthorized access, privacy breaches, or violations of federal laws governing student privacy and data security.\n\nOverall, a key recurring theme is borrower frustration with servicing errors, mismanagement, and lack of clarity in their student loan account management.\n\nIf you need a specific concise answer:  \n**The most common issue with 

In [53]:
semantic_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, it appears that several complaints were marked as "Closed with explanation" and the response indicated they were handled in a timely manner, with responses marked as "Yes" for timely response. For example:\n\n- Complaint against Nelnet (ID 13331376) received on 05/04/25 was responded to with "Closed with explanation" and marked as timely.\n- Complaint against Maximus Federal Services (ID 13207537) received on 04/28/25 was also responded to and marked as timely.\n- Complaint against EdFinancial Services (ID 13281034) received on 05/01/25 was answered and marked as timely.\n- Similarly, other complaints regarding payment handling and disputes were responded to within the expected timeframe.\n\nThere are no indications in the provided data that complaints did not get handled in a timely manner. All responses are marked as "Yes" for timely response, and no complaints are noted as unaddressed or delayed.\n\nTherefore, based on this information, **no compla

In [54]:
semantic_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including issues with loan servicing, mismanagement or delays in processing payments, lack of transparency and poor communication from lenders or servicers, and legal or administrative complications such as disputes over the legitimacy of the debt, improper reporting, or breaches of privacy laws. Some borrowers also faced difficulties due to procedural delays or deliberately stalling by loan servicers, which discouraged them from continuing efforts to repay.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

### ✅ ANSWER
When dealing with short, repetitive sentences like FAQs, semantic chunking might struggle with over-segmentation due to high similarity between adjacent Q&A pairs, potentially creating very small chunks or incorrectly grouping unrelated questions that use similar terminology. To address this, I would adjust the algorithm by using more conservative thresholding (higher percentile), implementing structural rules to keep Q&A pairs as atomic units, and adding topic-based pre-processing to group related FAQs before chunking. Additionally, I would set minimum chunk sizes to prevent single-sentence chunks and use FAQ section headers as natural break points. This hybrid approach combining semantic similarity with structural awareness would better preserve the coherence and usefulness of FAQ content.


# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

### ✅ ANSWER / SUMMARY OF RESULTS

Let's first take a look at cost and latency for each retrieval strategy using LangSmith:
- Latency: 
- Cost from lowest to greatest: BM25,  Naive, 
![RAGAS Comparison](ragas_comparison.png)

![Comparison](langsmith_eval_comparison.png)






In [55]:
import os
import getpass

In [56]:
os.environ["LANGSMITH_PROJECT"] = "Advanced RAG"
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter your Langsmith API Key:")
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [71]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


docs = loan_complaint_data
generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs[:5], testset_size=10)

Applying SummaryExtractor:   0%|          | 0/3 [00:00<?, ?it/s]

2025-07-25 15:24:55 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:24:55 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:24:56 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Applying CustomNodeFilter:   0%|          | 0/5 [00:00<?, ?it/s]

2025-07-25 15:24:56 - ragas.testset.transforms.filters - WARNING - Node 2967d11d-2a40-4a08-8f2d-ebf20f36a348 does not have a summary. Skipping filtering.
2025-07-25 15:24:56 - ragas.testset.transforms.filters - WARNING - Node 3a0f93ce-66d1-4167-8ba3-425aad517159 does not have a summary. Skipping filtering.
2025-07-25 15:24:57 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:24:57 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:24:58 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/13 [00:00<?, ?it/s]

...

2025-07-25 15:25:03 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:25:04 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:25:04 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

2025-07-25 15:25:04 - ragas.testset.synthesizers.multi_hop.abstract - INFO - found 4 clusters
2025-07-25 15:25:05 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:25:05 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:25:06 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:25:07 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:25:07 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:25:08 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:25:08 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:25:09 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

2025-07-25 15:25:13 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:25:13 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:25:14 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:25:14 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:25:14 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:25:15 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:25:15 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:25:15 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:25:15 - httpx - INFO - HTTP Request: POST https://

In [90]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,How did Nelnet handle the re-amortization of f...,[The federal student loan COVID-19 forbearance...,Payments on federal student loans serviced by ...,single_hop_specifc_query_synthesizer
1,Why Aidvantage give me wrong Income-Driven Rep...,[I submitted my annual Income-Driven Repayment...,Aidvantage gave you a wrong Income-Driven Repa...,single_hop_specifc_query_synthesizer
2,How does a violation of FERPA involving compro...,[My personal and financial data was compromise...,The context states that personal and financial...,single_hop_specifc_query_synthesizer
3,why studentaid.gov say nelnet my issuer but ne...,"[According to Studentaid.gov, Im to get an ema...","According to Studentaid.gov, you are supposed ...",single_hop_specifc_query_synthesizer
4,How has the resumption of federal loan payment...,[Since the resumption of federal loan payments...,"Since the resumption of federal loan payments,...",single_hop_specifc_query_synthesizer
5,How has student loan servicing by Nelnet contr...,[<1-hop>\n\nThe federal student loan COVID-19 ...,Student loan servicing by Nelnet has contribut...,multi_hop_abstract_query_synthesizer
6,"How has student loan servicing confusion, part...",[<1-hop>\n\nThe federal student loan COVID-19 ...,After the end of the federal student loan COVI...,multi_hop_abstract_query_synthesizer
7,How can delays in re-amortizing federal studen...,[<1-hop>\n\nThe federal student loan COVID-19 ...,Delays in re-amortizing federal student loan p...,multi_hop_abstract_query_synthesizer
8,How has the end of the forbearance period led ...,[<1-hop>\n\nThe federal student loan COVID-19 ...,The end of the federal student loan COVID-19 f...,multi_hop_abstract_query_synthesizer
9,How has loan servicing by Nelnet and the inves...,[<1-hop>\n\nThe federal student loan COVID-19 ...,Loan servicing by Nelnet resulted in delayed r...,multi_hop_abstract_query_synthesizer


In [ ]:
from langsmith import Client

client = Client()

dataset_name = "Loan Synthetic Data - Advanced Retrieval 3"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Loan Synthetic Data - Advanced Retrieval 3"
)

In [107]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

In [117]:
naive_retrieval_chain.name = "naive_retrieval"
bm25_retrieval_chain.name = "bm25_retrieval"
contextual_compression_retrieval_chain.name = "contextual_compression_retrieval"
multi_query_retrieval_chain.name = "multi_query_retrieval"
parent_document_retrieval_chain.name = "parent_document_retrieval"
ensemble_retrieval_chain.name = "ensemble_retrieval"
semantic_retrieval_chain.name = "semantic_retrieval"

retrievers = [
    naive_retrieval_chain,
    bm25_retrieval_chain,
    contextual_compression_retrieval_chain,
    multi_query_retrieval_chain,
    parent_document_retrieval_chain,
    ensemble_retrieval_chain,
]



In [73]:
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [74]:
import logging

# Configure logging to show INFO level messages
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

In [ ]:
import uuid
from langsmith.evaluation import evaluate

def langsmith_evaluation(rag_chain, dataset_name):
    logger.info(f"Starting LangSmith evaluation for chain: {rag_chain.__class__.__name__}")
    logger.info(f"Using dataset: {dataset_name}")
    id = str(uuid.uuid4())
    
    # Define a wrapper function that handles the input format
    def chain_invoke(inputs):
        # Extract just the question string from the inputs
        if isinstance(inputs, dict) and "question" in inputs:
            query = inputs["question"]
        else:
            query = str(inputs)
        
        return rag_chain.invoke({"question": query})
    
    try:
        logger.info("Starting LangSmith evaluation")
        logging.info(f"{rag_chain.name}_{id}")
        evaluate(
            chain_invoke,
            data=dataset_name,
            evaluators=[],
            experiment_prefix=rag_chain.name
            metadata={"revision_id": f"{rag_chain.name}_{id}"},
        )
        logger.info("LangSmith evaluation completed successfully")
    except Exception as e:
        logger.error(f"Error during LangSmith evaluation: {str(e)}")
        raise

In [91]:
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate as ragas_evaluate, RunConfig, EvaluationDataset
from ragas.llms import LangchainLLMWrapper
import copy

custom_run_config = RunConfig(timeout=360)
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

def ragas_evaluation(rag_chain, dataset):
  print(f"Starting RAGAS evaluation for chain: {rag_chain.__class__.__name__}")
  
  working_dataset = copy.deepcopy(dataset)
  print(f"Processing {len(working_dataset)} samples")

  for i, test_row in enumerate(working_dataset):
    print(f"Processing sample {i+1}/{len(working_dataset)}")
    print(f"eval_sample: {test_row.eval_sample}")
    print(f"type: {type(test_row.eval_sample)}")
    print(f"Question: {test_row.eval_sample.user_input}")
    print(f"Reference: {test_row.eval_sample.reference}")
    print(f"Reference Contexts: {test_row.eval_sample.reference_contexts}")
    response = rag_chain.invoke({"question" : test_row.eval_sample.user_input})
    test_row.eval_sample.response = response["response"].content
    test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
    print(f"Response: {test_row.eval_sample.response}")
    print(f"Retrieved Contexts ({type(test_row.eval_sample.retrieved_contexts)}): {test_row.eval_sample.retrieved_contexts}")
    print(f"Retrieved {len(test_row.eval_sample.retrieved_contexts)} contexts")

  print("Creating evaluation dataset")
  try:
    evaluation_dataset = EvaluationDataset.from_pandas(working_dataset.to_pandas())
  except Exception as e:
    print(f"Error creating evaluation dataset: {e}")
    return None

  print("Running RAGAS evaluation metrics")
  result = ragas_evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
  )
  
  print("RAGAS evaluation complete")
  print(f"Results: {result}")
  
  return result

In [ ]:
metrics = {}
for chain in retrievers:
  ragas_metrics = ragas_evaluation(chain, dataset)
  metrics[chain.name] = ragas_metrics

Starting RAGAS evaluation for chain: RunnableSequence
Processing 11 samples
Processing sample 1/11
eval_sample: user_input='why my payment go up so much after federal student loan COVID-19 forbearance program end?' retrieved_contexts=None reference_contexts=["The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers."] response=None multi_responses=None reference='The federal student loan COVID-19 forbearance pro

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Response: The significant increase in your federal student loan payments after the COVID-19 forbearance program ended is likely due to the loans not being re-amortized or adjusted once the forbearance concluded. During forbearance, payments are typically paused or reduced, and interest may accrue depending on the specific circumstances. When the forbearance ended, the repayment terms may not have been automatically recalculated to reflect the temporary pause, resulting in larger monthly payments.

Some complaints in the context indicate that re-amortization or adjustment efforts were delayed or did not occur promptly, causing payments to nearly double or increase substantially. For example, one borrower reported that their payment nearly doubled from $180 to $360 because the loans were not re-amortized in a timely manner after the forbearance ended. Additionally, in some cases, interest may have continued to accrue during forbearance, further increasing the total amount owed and leadin

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Response: Aidvantage assigned you an incorrect monthly payment amount on your Income-Driven Repayment (IDR) plan despite your income qualifying for a lower payment because their system appears to have calculated or billed an amount that is not consistent with your documented income and the applicable repayment plan. Despite your timely recertification and submission of the correct income information, they continued to bill an amount that was not legally or mathematically possible based on your income, and they did not process your application promptly, which may have contributed to this discrepancy.

In response to your inquiries, Aidvantage placed you in a two-month administrative forbearance and then continued to bill the same incorrect amount without processing your income-driven plan application or requesting further documentation, even though you repeatedly emphasized that your income qualifies for a lower payment. When you pressed for clarification, they claimed that your applica

...

RAGAS evaluation complete
Results: {'context_recall': 0.8203, 'faithfulness': 0.7286, 'factual_correctness(mode=f1)': 0.4445, 'answer_relevancy': 0.5861, 'context_entity_recall': 0.3961, 'noise_sensitivity(mode=relevant)': 0.1778}
Starting RAGAS evaluation for chain: RunnableSequence
Processing 11 samples
Processing sample 1/11
eval_sample: user_input='why my payment go up so much after federal student loan COVID-19 forbearance program end?' retrieved_contexts=None reference_contexts=["The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.cohere.com/v2/rerank "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Response: The increase in your federal student loan payments after the COVID-19 forbearance program ended is likely due to the loans being re-amortized based on the original loan terms. During the forbearance period, payments were temporarily paused or reduced, and interest may not have accrued or was accrued differently depending on the type of forbearance. 

Once the forbearance ended, the loans needed to be recalculated (re-amortized) to include any outstanding interest and to set new monthly payments based on the current loan balance and original repayment schedule. If this re-amortization was delayed or not handled promptly, your payments could have increased significantly—sometimes nearly doubling—since the loan was recalculated to cover the full amount owed over the remaining loan term.

Additionally, some issues noted in complaints include loans going into interest-only payments or balances increasing despite payments, which can also contribute to higher monthly payments after 

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.cohere.com/v2/rerank "HTTP/1.1 200 OK"


KeyboardInterrupt: 

In [104]:
for chain in retrievers:
  langsmith_evaluation(chain, dataset_name)

2025-07-25 16:40:21 - __main__ - INFO - Starting LangSmith evaluation for chain: RunnableSequence
2025-07-25 16:40:21 - __main__ - INFO - Using dataset: Loan Synthetic Data - Advanced Retrieval 2
2025-07-25 16:40:21 - __main__ - INFO - Starting LangSmith evaluation
2025-07-25 16:40:21 - root - INFO - naive_retrieval_d7211a1b-fd39-4c6a-ab2c-24b60f15a158


View the evaluation results for experiment: 'naive_retrieval-b67626b4' at:
https://smith.langchain.com/o/c2cfcbd8-d5df-509f-8f0e-973ec8ab5a6b/datasets/fc4e91e3-b3a9-40fd-920d-ffdbb53f6f68/compare?selectedSessions=235ef012-5c09-4bde-8d76-084eea8668b7




0it [00:00, ?it/s]

2025-07-25 16:40:22 - __main__ - INFO - Query: Why is it important to investigate loan servicer practices like Nelnet's handling of re-amortization after the COVID-19 forbearance and Aidvantage's billing for IDR plans before processing applications, and how do these issues affect borrowers?
2025-07-25 16:40:22 - __main__ - INFO - Using chain invoke method for naive_retrieval
2025-07-25 16:40:22 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 16:40:26 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:40:26 - __main__ - INFO - Query: How has loan servicing by Nelnet and the investigation into loan servicer practices affected borrowers facing payment recalculations and billing errors after the COVID-19 forbearance?
2025-07-25 16:40:26 - __main__ - INFO - Using chain invoke method for naive_retrieval
2025-07-25 16:40:26 - httpx - INFO - HTTP Request: POST https://api.openai.co

View the evaluation results for experiment: 'bm25_retrieval-9208348f' at:
https://smith.langchain.com/o/c2cfcbd8-d5df-509f-8f0e-973ec8ab5a6b/datasets/fc4e91e3-b3a9-40fd-920d-ffdbb53f6f68/compare?selectedSessions=21442495-0ec2-43e9-8d05-b79a17dddef3




...

0it [00:00, ?it/s]

2025-07-25 16:44:51 - __main__ - INFO - Query: Why is it important to investigate loan servicer practices like Nelnet's handling of re-amortization after the COVID-19 forbearance and Aidvantage's billing for IDR plans before processing applications, and how do these issues affect borrowers?
2025-07-25 16:44:51 - __main__ - INFO - Using chain invoke method for None
2025-07-25 16:44:51 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 16:44:54 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:44:54 - __main__ - INFO - Query: How has loan servicing by Nelnet and the investigation into loan servicer practices affected borrowers facing payment recalculations and billing errors after the COVID-19 forbearance?
2025-07-25 16:44:54 - __main__ - INFO - Using chain invoke method for None
2025-07-25 16:44:54 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/

View the evaluation results for experiment: 'ensemble_retrieval-16056aec' at:
https://smith.langchain.com/o/c2cfcbd8-d5df-509f-8f0e-973ec8ab5a6b/datasets/fc4e91e3-b3a9-40fd-920d-ffdbb53f6f68/compare?selectedSessions=63b8b654-76bd-4e31-a518-5172165cf161




0it [00:00, ?it/s]

2025-07-25 16:45:23 - __main__ - INFO - Query: Why is it important to investigate loan servicer practices like Nelnet's handling of re-amortization after the COVID-19 forbearance and Aidvantage's billing for IDR plans before processing applications, and how do these issues affect borrowers?
2025-07-25 16:45:23 - __main__ - INFO - Using chain invoke method for ensemble_retrieval
2025-07-25 16:45:24 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 16:45:24 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 16:45:24 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 16:45:25 - httpx - INFO - HTTP Request: POST https://api.cohere.com/v2/rerank "HTTP/1.1 200 OK"
2025-07-25 16:45:26 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:45:26 - langchain.retrievers.multi_query - INFO 

In [119]:
langsmith_evaluation(parent_document_retrieval_chain, dataset_name)

2025-07-25 21:18:57 - __main__ - INFO - Starting LangSmith evaluation for chain: RunnableSequence
2025-07-25 21:18:57 - __main__ - INFO - Using dataset: Loan Synthetic Data - Advanced Retrieval 2
2025-07-25 21:18:57 - __main__ - INFO - Starting LangSmith evaluation
2025-07-25 21:18:57 - root - INFO - parent_document_retrieval_2d386a94-8f95-4d4e-afce-d223471653b0


View the evaluation results for experiment: 'parent_document_retrieval-56aadc81' at:
https://smith.langchain.com/o/c2cfcbd8-d5df-509f-8f0e-973ec8ab5a6b/datasets/fc4e91e3-b3a9-40fd-920d-ffdbb53f6f68/compare?selectedSessions=a6775a81-148a-410d-86d2-24347376cb8d




0it [00:00, ?it/s]

2025-07-25 21:18:58 - __main__ - INFO - Query: Why is it important to investigate loan servicer practices like Nelnet's handling of re-amortization after the COVID-19 forbearance and Aidvantage's billing for IDR plans before processing applications, and how do these issues affect borrowers?
2025-07-25 21:18:58 - __main__ - INFO - Using chain invoke method for parent_document_retrieval
2025-07-25 21:18:59 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 21:19:07 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 21:19:08 - __main__ - INFO - Query: How has loan servicing by Nelnet and the investigation into loan servicer practices affected borrowers facing payment recalculations and billing errors after the COVID-19 forbearance?
2025-07-25 21:19:08 - __main__ - INFO - Using chain invoke method for parent_document_retrieval
2025-07-25 21:19:08 - httpx - INFO - HTTP Request: POST h

In [92]:
naive_retrieval_ragas_metrics = ragas_evaluation(naive_retrieval_chain, dataset)
naive_retrieval_ragas_metrics

Starting RAGAS evaluation for chain: RunnableSequence
Processing 11 samples
Processing sample 1/11
eval_sample: user_input='How did Nelnet handle the re-amortization of federal student loan payments after the COVID-19 forbearance ended?' retrieved_contexts=None reference_contexts=["The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers."] response=None multi_responses=None reference='Payments on federal studen

2025-07-25 15:47:10 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 15:47:12 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Response: Based on the information provided, Nelnet handled the re-amortization of federal student loan payments after the COVID-19 forbearance ended with some delays. While the forbearance periods were supposed to end automatically and borrowers should have been notified to help them adjust to new payment amounts, complaints indicate that payments were not re-amortized or adjusted immediately after the forbearance ended. In some cases, payments increased significantly, nearly doubling, without prompt re-amortization or clear communication from Nelnet.

Several complaints highlight that Nelnet's timing in re-amortizing payments was recent, and that borrowers experienced challenges due to lack of prior notification, incorrect account handling, or delays in updating payment amounts. Additionally, there are reports of missed notifications regarding the end of forbearance, leading to issues such as unrecognized delinquency reporting, credit score impacts, and confusion over payment obligat

2025-07-25 15:47:12 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 15:47:15 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Response: I'm sorry to hear about the issues you're experiencing with Aidvantage's Income-Driven Repayment (IDR) calculations. Based on the context provided, here are some possible reasons why you might be receiving an incorrect payment amount despite submitting all necessary income documentation:

1. Processing Delays or Administrative Issues: Several complaints indicate that Aidvantage has experienced delays or administrative hold-ups in processing IDR recertifications, which can lead to incorrect billing or defaulting to a standard repayment plan instead of the correct IDR plan.

2. Unprocessed Applications: Some borrowers mention that their applications are marked as "processing" or delayed due to government holds or court orders, preventing the system from recalculating payments based on income.

3. Billing Prior to Application Processing: There are complaints where borrowers were billed the wrong amount even before their IDR application was processed, suggesting possible miscommu

...

Response: Investigating loan servicer practices, such as Nelnet's handling of re-amortization after COVID-19 forbearance and Aidvantage's billing practices for IDR plans, is crucial because these practices directly impact borrowers' financial well-being, credit reports, and access to protections and benefits under federal student aid programs. 

Why is it important to investigate these practices?

1. **Ensuring Proper Notification and Communication:** The complaints highlight failures in providing borrowers with timely and clear notifications about when forbearance ends or when repayment resumes. Lack of proper communication can lead to unintentional delinquency, damage to credit scores, and missed opportunities for borrowers to manage their loans effectively.

2. **Protecting Against Improper Reporting:** Several cases demonstrate that loans were reported as delinquent or past due without proper notice or when the accounts were in government-sanctioned forbearance or deferment. Invest

Evaluating:   0%|          | 0/66 [00:00<?, ?it/s]

2025-07-25 15:47:49 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:47:49 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:47:49 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:47:49 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:47:49 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:47:49 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:47:49 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 15:47:49 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 15:47:49 - httpx - INFO - HTTP Request: POST https://api.openai.c

RAGAS evaluation complete
Results: {'context_recall': 0.9104, 'faithfulness': 0.8831, 'factual_correctness(mode=f1)': 0.6427, 'answer_relevancy': 0.7818, 'context_entity_recall': 0.4592, 'noise_sensitivity(mode=relevant)': 0.3029}


{'context_recall': 0.9104, 'faithfulness': 0.8831, 'factual_correctness(mode=f1)': 0.6427, 'answer_relevancy': 0.7818, 'context_entity_recall': 0.4592, 'noise_sensitivity(mode=relevant)': 0.3029}

In [93]:
bm25_retrieval_ragas_metrics = ragas_evaluation(bm25_retrieval_chain, dataset)
bm25_retrieval_ragas_metrics

Starting RAGAS evaluation for chain: RunnableSequence
Processing 11 samples
Processing sample 1/11
eval_sample: user_input='How did Nelnet handle the re-amortization of federal student loan payments after the COVID-19 forbearance ended?' retrieved_contexts=None reference_contexts=["The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers."] response=None multi_responses=None reference='Payments on federal studen

2025-07-25 15:55:54 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Response: Nelnet handled the re-amortization of federal student loan payments after the COVID-19 forbearance ended by delaying the process and not re-amortizing the payments immediately once the forbearance concluded. According to a consumer complaint, payments were not updated to reflect the new payment amounts until very recently, leading to a significant increase in monthly payments—from approximately $180 to nearly $360. The delay in re-amortization meant that borrowers experienced an unexpected and nearly doubled payment amount, which could impact those with less financial flexibility. Nelnet's response to the complaint was to close the issue with an explanation, indicating that the re-amortization did not occur promptly after the forbearance period ended.

If you need specific details or assistance, I can help further.
Retrieved Contexts (<class 'list'>): ["The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my fed

2025-07-25 15:55:57 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Response: I'm sorry you're experiencing issues with Aidvantage providing incorrect Income-Driven Repayment (IDR) payment amounts despite submitting all required documentation and having low income. Based on the complaints documented, there are several possible reasons why this might be happening:

1. Processing Delays or Errors: Some borrowers report that their recertification applications are marked as processing or are delayed due to external factors, leading to incorrect billing amounts. In some cases, Aidvantage has indicated that their system has systemic errors or that applications haven't been fully processed, yet bills are issued based on incomplete information.

2. Lack of Communication or Follow-up: Borrowers mention that Aidvantage sometimes does not request additional documentation or follow up for income verification, which may result in default or incorrect payments. When applications are delayed or not fully processed, the system may default to a default payment amount r

...

Response: Investigating loan servicer practices like Nelnet's handling of re-amortization after COVID-19 forbearance and Aidvantage's billing for IDR plans is important because these practices directly impact borrowers' ability to access, apply for, and benefit from repayment options designed to make student debt more manageable. 

These issues affect borrowers in several ways:

1. **Access and Transparency:** If servicers are not processing applications correctly or are withholding information, borrowers may be unable to enroll in plans like Income-Driven Repayment (IDR), which can reduce monthly payments and provide loan forgiveness options. Lack of transparency can lead to confusion and a sense of mistrust.

2. **Interest Accrual and Financial Burden:** Mishandling of re-amortization or billing can result in unnecessary interest accrual or incorrect billing, increasing the overall debt and making repayment more burdensome.

3. **Eligibility and Benefits Loss:** Errors or delays in p

Evaluating:   0%|          | 0/66 [00:00<?, ?it/s]

2025-07-25 15:56:19 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:56:19 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:56:19 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:56:19 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:56:19 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:56:19 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:56:19 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:56:19 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 15:56:19 - httpx - INFO - HTTP Request: POST https://

RAGAS evaluation complete
Results: {'context_recall': 0.7701, 'faithfulness': 0.8710, 'factual_correctness(mode=f1)': 0.5073, 'answer_relevancy': 0.7882, 'context_entity_recall': 0.3742, 'noise_sensitivity(mode=relevant)': 0.3867}


{'context_recall': 0.7701, 'faithfulness': 0.8710, 'factual_correctness(mode=f1)': 0.5073, 'answer_relevancy': 0.7882, 'context_entity_recall': 0.3742, 'noise_sensitivity(mode=relevant)': 0.3867}

In [96]:
contextual_compression_retrieval_ragas_metrics = ragas_evaluation(contextual_compression_retrieval_chain, dataset)
contextual_compression_retrieval_ragas_metrics

Starting RAGAS evaluation for chain: RunnableSequence
Processing 11 samples
Processing sample 1/11
eval_sample: user_input='How did Nelnet handle the re-amortization of federal student loan payments after the COVID-19 forbearance ended?' retrieved_contexts=None reference_contexts=["The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers."] response=None multi_responses=None reference='Payments on federal studen

2025-07-25 16:02:53 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 16:02:53 - httpx - INFO - HTTP Request: POST https://api.cohere.com/v2/rerank "HTTP/1.1 200 OK"
2025-07-25 16:02:59 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Response: Based on the information provided, Nelnet did not handle the re-amortization of federal student loan payments immediately after the COVID-19 forbearance ended. Although the forbearance program ended in a certain month, payments were not re-amortized or adjusted to reflect the new payment amounts until very recently. As a result, for some borrowers, the new payment amounts, which nearly doubled the previous payments, only took effect after a delay. This delay led to situations where borrowers faced sudden increases in their monthly payments, and for some, the re-amortization did not occur promptly, causing concern about the impact on their finances. Additionally, there were complaints about lack of proper notification and communication regarding the end of forbearance and commencement of repayment, which may have contributed to confusion and delinquency issues.
Retrieved Contexts (<class 'list'>): ["The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. How

2025-07-25 16:02:59 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 16:02:59 - httpx - INFO - HTTP Request: POST https://api.cohere.com/v2/rerank "HTTP/1.1 200 OK"
2025-07-25 16:03:35 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Response: I'm sorry you're experiencing issues with your Income-Driven Repayment (IDR) payments through Aidvantage. Based on the complaints provided in the context, there are a few potential reasons why you might be receiving incorrect payment amounts despite providing all your income information:

1. **Processing Delays or Errors:** Several complaints mention that even after submitting recertification, their applications were marked as "processing" and not yet finalized. During this period, the correct IDR payment amounts might not be applied, leading to incorrect billing.

2. **Billing Before Final Processing:** Some borrowers reported being billed for amounts based on recalculations prior to their applications being fully processed and approved. This can cause mismatched or incorrect payments.

3. **Lack of Communication or Clarification:** There are instances where Aidvantage did not request additional documentation or clarify how payments were calculated, leading to discrepancies.

...

Response: Investigating loan servicer practices like Nelnet's handling of re-amortization after COVID-19 forbearance and Aidvantage's billing for Income-Driven Repayment (IDR) plans is crucial because these practices directly impact borrowers' financial well-being and credit reports. 

Why is this important?

1. **Protecting Borrowers' Rights and Transparency:** Proper notification and communication ensure borrowers are aware of their repayment status, upcoming bills, and any changes to their loan terms. For example, in the case of Nelnet, failure to notify borrowers about the end of forbearance led to premature delinquency reporting, damaging credit scores without proper warning.

2. **Ensuring Accurate Reporting:** Incorrect or delayed reporting of delinquencies can adversely affect credit scores and future borrowing opportunities. Investigating these practices helps enforce compliance with the Fair Credit Reporting Act and Department of Education guidelines.

3. **Fair Treatment and

Evaluating:   0%|          | 0/66 [00:00<?, ?it/s]

2025-07-25 16:04:12 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:04:12 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:04:12 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:04:12 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:04:13 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 16:04:13 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 16:04:13 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:04:13 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:04:13 - httpx - INFO - HTTP Request: POST https://api.openai.c

RAGAS evaluation complete
Results: {'context_recall': 0.8437, 'faithfulness': 0.8437, 'factual_correctness(mode=f1)': 0.6509, 'answer_relevancy': 0.7686, 'context_entity_recall': 0.5130, 'noise_sensitivity(mode=relevant)': 0.2740}


{'context_recall': 0.8437, 'faithfulness': 0.8437, 'factual_correctness(mode=f1)': 0.6509, 'answer_relevancy': 0.7686, 'context_entity_recall': 0.5130, 'noise_sensitivity(mode=relevant)': 0.2740}

In [97]:
multi_query_retrieval_ragas_metrics = ragas_evaluation(multi_query_retrieval_chain, dataset)
multi_query_retrieval_ragas_metrics

Starting RAGAS evaluation for chain: RunnableSequence
Processing 11 samples
Processing sample 1/11
eval_sample: user_input='How did Nelnet handle the re-amortization of federal student loan payments after the COVID-19 forbearance ended?' retrieved_contexts=None reference_contexts=["The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers."] response=None multi_responses=None reference='Payments on federal studen

2025-07-25 16:08:34 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:08:34 - langchain.retrievers.multi_query - INFO - Generated queries: ['1. What procedures did Nelnet follow when re-amortizing federal student loan payments after the COVID-19 forbearance period concluded?', '2. How did Nelnet implement the re-amortization process for federal student loans once the COVID-19 forbearance was lifted?', '3. In what ways did Nelnet manage the adjustment and re-amortization of federal student loans post-COVID-19 forbearance?']
2025-07-25 16:08:35 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 16:08:35 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 16:08:35 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 16:08:37 - httpx - INFO - HTTP Request: POST https://api.openai.co

Response: Based on the provided complaints and context, Nelnet's handling of re-amortization of federal student loan payments after the COVID-19 forbearance ended appears to have been delayed or inadequate. Several consumers reported that their payments were not re-amortized immediately once the forbearance ended, leading to significantly increased payment amounts starting with subsequent payments. In some cases, borrowers received little to no notification about the resumption of payments, leading to unexpected delinquency reports, drops in credit scores, and confusion about their loan status.

While Nelnet's official public responses are not provided in the complaints, the consumer narratives suggest that Nelnet did not promptly re-amortize loans or adequately notify borrowers when the COVID-19 forbearance ended. Several consumers indicated that payments only recently started reflecting the new terms, and there were issues with communication, notification, and timely adjustments.

In

2025-07-25 16:08:38 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:08:38 - langchain.retrievers.multi_query - INFO - Generated queries: ['What are the possible reasons Aidvantage calculated an incorrect Income-Driven Repayment amount despite submitting all necessary documents and having a low income?', 'How can I troubleshoot why Aidvantage provided an incorrect income-based repayment plan after I submitted my income information and proof?', 'Why did Aidvantage assign me the wrong Income-Driven Repayment payment amount even though I provided accurate income details and my income is low?']
2025-07-25 16:08:38 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 16:08:39 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 16:08:40 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07

Response: I'm sorry you're experiencing difficulties with Aidvantage's income-driven repayment (IDR) amounts. Based on the complaints data, people have faced issues such as:

- Sending all required documentation and having their income and application acknowledged, but still being billed incorrect, often higher, payment amounts not aligned with their low income.
- Being told their application has not been processed, yet being billed at incorrect amounts.
- Delays or failures in applying the correct IDR payment based on income, even after submission.
- Discrepancies between communication and actual billing or loan status, sometimes leading to default or incorrect repayment plans.

These issues may be caused by administrative errors, processing delays, or miscommunication within the servicer's system. Often, even when borrowers have completed the recertification process and qualify for lower payments, the system hasn't updated correctly, resulting in overcharged amounts.

If you've done 

...

Response: Investigation into loan servicer practices like Nelnet's handling of re-amortization after COVID-19 forbearance and Aidvantage's billing for Income-Driven Repayment (IDR) plans is crucial because these issues directly impact borrowers' financial well-being and trust in the federal student loan system.

Why is it important?  
- **Protect Borrowers from Unfair or Harmful Practices:** Proper re-amortization ensures that borrowers are not abruptly subjected to higher monthly payments, which can cause financial hardship. Proper notification about loan status and billing helps borrowers avoid unintentional delinquencies or default.  
- **Ensure Compliance with Federal Laws and Regulations:** Servicers are obligated under federal regulations and consumer protection laws to provide accurate information, timely notifications, and fair treatment. Investigating practices helps verify adherence and address systemic issues.  
- **Prevent Financial and Credit Harm:** Mishandling these proc

Evaluating:   0%|          | 0/66 [00:00<?, ?it/s]

2025-07-25 16:10:30 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:10:30 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:10:30 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:10:31 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:10:31 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:10:31 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:10:31 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:10:31 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 16:10:31 - httpx - INFO - HTTP Request: POST https://api.op

RAGAS evaluation complete
Results: {'context_recall': 0.9558, 'faithfulness': 0.9202, 'factual_correctness(mode=f1)': 0.5236, 'answer_relevancy': 0.6052, 'context_entity_recall': 0.4630, 'noise_sensitivity(mode=relevant)': 0.3751}


{'context_recall': 0.9558, 'faithfulness': 0.9202, 'factual_correctness(mode=f1)': 0.5236, 'answer_relevancy': 0.6052, 'context_entity_recall': 0.4630, 'noise_sensitivity(mode=relevant)': 0.3751}

In [99]:
parent_document_retrieval_ragas_metrics = ragas_evaluation(parent_document_retrieval_chain, dataset)
parent_document_retrieval_ragas_metrics

Starting RAGAS evaluation for chain: RunnableSequence
Processing 11 samples
Processing sample 1/11
eval_sample: user_input='How did Nelnet handle the re-amortization of federal student loan payments after the COVID-19 forbearance ended?' retrieved_contexts=None reference_contexts=["The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers."] response=None multi_responses=None reference='Payments on federal studen

2025-07-25 16:19:35 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 16:19:36 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Response: Based on the provided complaints, Nelnet's handling of the re-amortization of federal student loan payments after the COVID-19 forbearance ended appears to have been delayed and problematic. Borrowers reported that their payments were not re-amortized promptly once the forbearance ended, leading to substantial increases in their monthly payments—sometimes nearly doubling—and causing stress and financial hardship. 

Specifically, the complaints indicate that Nelnet did not re-calculate or lower the monthly payments as expected after the forbearance period concluded. In some cases, payments were only recently re-amortized, resulting in higher obligations for borrowers. This lack of timely adjustment was considered an oversight by borrowers and was viewed as unfair, especially for those unprepared for the sudden increase. The issues also extended to poor communication, with some borrowers not receiving adequate notices about when payments would restart or how much they would be.

2025-07-25 16:19:37 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 16:19:58 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Response: I'm sorry to hear about your difficulties with Aidvantage regarding your Income-Driven Repayment (IDR) plan. Based on the information provided, there are a few possible reasons why your payment amount might be incorrect even after you've submitted all your income information:

1. **Processing Delays or Errors:** The complaint mentions that some applications and recertifications are marked as "processing" and may not have been fully processed or recalculated yet. If Aidvantage has not completed processing your application, they might be basing your payment on outdated or incorrect information.

2. **System or Administrative Mistakes:** There are reports of incorrect billing amounts being issued despite documentation showing low income, which could indicate administrative or system errors.

3. **Lack of Communication or Clarification:** In some cases, Aidvantage has not requested additional necessary documentation or properly communicated with borrowers about the status of thei

...

Response: Investigating loan servicer practices, such as Nelnet's handling of re-amortization after COVID-19 forbearance and Aidvantage's billing for Income-Driven Repayment (IDR) plans, is important because these practices directly impact borrowers’ financial well-being, credit reports, and understanding of their loan status. 

When servicers fail to properly notify borrowers about changes in their loan status, such as the end of forbearance or the approval of repayment plans, it can lead to premature delinquency reports, missed or incorrect payments, and negative impacts on credit scores. For example, Nelnet has been reported to inaccurately report loans as delinquent despite the accounts being in forbearance, causing unnecessary credit damage and confusion. Similarly, Aidvantage's failure to process IDR plan applications for over a year leaves borrowers in financial distress and prevents them from accessing income-based repayment options that could alleviate hardship.

These issues 

Evaluating:   0%|          | 0/66 [00:00<?, ?it/s]

2025-07-25 16:20:31 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:20:31 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:20:31 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:20:31 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:20:31 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:20:31 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:20:31 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:20:31 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 16:20:32 - httpx - INFO - HTTP Request: POST https://api.op

RAGAS evaluation complete
Results: {'context_recall': 0.7801, 'faithfulness': 0.8026, 'factual_correctness(mode=f1)': 0.6227, 'answer_relevancy': 0.6983, 'context_entity_recall': 0.4857, 'noise_sensitivity(mode=relevant)': 0.3171}


{'context_recall': 0.7801, 'faithfulness': 0.8026, 'factual_correctness(mode=f1)': 0.6227, 'answer_relevancy': 0.6983, 'context_entity_recall': 0.4857, 'noise_sensitivity(mode=relevant)': 0.3171}

In [100]:
ensemble_retrieval_ragas_metrics = ragas_evaluation(ensemble_retrieval_chain, dataset)
ensemble_retrieval_ragas_metrics

Starting RAGAS evaluation for chain: RunnableSequence
Processing 11 samples
Processing sample 1/11
eval_sample: user_input='How did Nelnet handle the re-amortization of federal student loan payments after the COVID-19 forbearance ended?' retrieved_contexts=None reference_contexts=["The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers."] response=None multi_responses=None reference='Payments on federal studen

2025-07-25 16:24:32 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 16:24:33 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 16:24:33 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 16:24:33 - httpx - INFO - HTTP Request: POST https://api.cohere.com/v2/rerank "HTTP/1.1 200 OK"
2025-07-25 16:24:36 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:24:36 - langchain.retrievers.multi_query - INFO - Generated queries: ['1. What specific procedures did Nelnet implement to re-amortize federal student loan payments once the COVID-19 forbearance period concluded?  ', '2. How did Nelnet manage the process of recalculating federal student loan payments following the end of COVID-19-related forbearance?  ', '3. In what ways did Nelnet address the re-amortization of federal stude

Response: Nelnet's handling of the re-amortization of federal student loan payments after the COVID-19 forbearance ended was delayed in some cases. The consumer complaints indicate that, although the forbearance period ended, payments were not re-amortized or restarted promptly. For example, one complainant noted that payments were not re-amortized until very recently, resulting in a significant increase in their monthly payment amount—nearly doubling from around $180 to $360 starting with a specific payment date. The complaint suggests that the re-amortization should have occurred automatically once the forbearance ended to mitigate the financial impact on borrowers.

In summary, Nelnet did not re-amortize federal student loan payments automatically immediately after the COVID-19 forbearance concluded in some cases, leading to delayed payments, unexpected payment increases, and borrower confusion. The company’s public response to these complaints was generally recorded as "None," but 

2025-07-25 16:24:39 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 16:24:39 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 16:24:39 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 16:24:40 - httpx - INFO - HTTP Request: POST https://api.cohere.com/v2/rerank "HTTP/1.1 200 OK"
2025-07-25 16:24:40 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:24:40 - langchain.retrievers.multi_query - INFO - Generated queries: ['1. Why does Aidvantage calculate incorrect Income-Driven Repayment amounts despite me submitting all my income documentation and having a low income?', '2. What are the possible reasons Aidvantage is providing me with incorrect repayment figures under Income-Driven Repayment plans, even though I’ve provided accurate income information and have a low income

Response: I'm sorry you're experiencing issues with your Income-Driven Repayment (IDR) payments and the incorrect billing from Aidvantage. Based on the complaints data, some common reasons why this might happen include:

- **Processing Delays:** Even after submitting your documentation on time, Aidvantage may not have processed your application promptly, leading to incorrect payment amounts being assigned.
- **System Errors:** There have been reports of Aidvantage billing borrowers before applications are fully processed or recalculating payments incorrectly based on outdated or unverified income information.
- **Lack of Communication:** Sometimes, Aidvantage responds that they haven't processed your application despite billing based on recalculations, indicating possible administrative or technical issues.
- **Failure to Request Additional Documentation:** If your income documentation isn't reviewed or verified properly, incorrect payments can be assigned.
- **Billing Despite Unproces

...

Response: Investigating loan servicer practices like Nelnet's handling of re-amortization after the COVID-19 forbearance and Aidvantage's billing for Income-Driven Repayment (IDR) plans before processing applications is crucial because these practices directly impact borrower rights, financial stability, and credit reports. 

**Why is it important?**
- **Protection of Borrowers' Rights:** Proper notification, transparency, and accurate reporting are mandated by laws and regulations (such as the Fair Credit Reporting Act and Department of Education guidelines). When servicers fail to notify borrowers about changes, interest accrual, or payment adjustments, borrowers can inadvertently miss payments, damage credit scores, or face unexpected financial burdens.
- **Preventing Mismanagement and Errors:** Ensuring proper re-amortization and accurate billing prevents errors like double billing, incorrect interest charges, or reported delinquencies that can have long-term negative effects on cr

Evaluating:   0%|          | 0/66 [00:00<?, ?it/s]

2025-07-25 16:26:43 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:26:43 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:26:44 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:26:44 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:26:44 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:26:44 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:26:44 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-07-25 16:26:44 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-07-25 16:26:45 - httpx - INFO - HTTP Request: POST https://api.op

RAGAS evaluation complete
Results: {'context_recall': 0.9537, 'faithfulness': 0.9330, 'factual_correctness(mode=f1)': 0.5327, 'answer_relevancy': 0.7838, 'context_entity_recall': 0.5481, 'noise_sensitivity(mode=relevant)': 0.5256}


{'context_recall': 0.9537, 'faithfulness': 0.9330, 'factual_correctness(mode=f1)': 0.5327, 'answer_relevancy': 0.7838, 'context_entity_recall': 0.5481, 'noise_sensitivity(mode=relevant)': 0.5256}

In [101]:
metrics_consolidated = {
    "naive_retrieval_chain" : naive_retrieval_ragas_metrics,
    "multi_query_retrieval_chain" : multi_query_retrieval_ragas_metrics,
    "bm25_retrieval_chain" : bm25_retrieval_ragas_metrics,
    "contextual_compression_retrieval_chain" : contextual_compression_retrieval_ragas_metrics,
    "parent_document_retrieval_chain" : parent_document_retrieval_ragas_metrics,
    "ensemble_retrieval_chain" : ensemble_retrieval_ragas_metrics,
}

In [102]:
metrics_consolidated

{'naive_retrieval_chain': {'context_recall': 0.9104, 'faithfulness': 0.8831, 'factual_correctness(mode=f1)': 0.6427, 'answer_relevancy': 0.7818, 'context_entity_recall': 0.4592, 'noise_sensitivity(mode=relevant)': 0.3029},
 'multi_query_retrieval_chain': {'context_recall': 0.9558, 'faithfulness': 0.9202, 'factual_correctness(mode=f1)': 0.5236, 'answer_relevancy': 0.6052, 'context_entity_recall': 0.4630, 'noise_sensitivity(mode=relevant)': 0.3751},
 'bm25_retrieval_chain': {'context_recall': 0.7701, 'faithfulness': 0.8710, 'factual_correctness(mode=f1)': 0.5073, 'answer_relevancy': 0.7882, 'context_entity_recall': 0.3742, 'noise_sensitivity(mode=relevant)': 0.3867},
 'contextual_compression_retrieval_chain': {'context_recall': 0.8437, 'faithfulness': 0.8437, 'factual_correctness(mode=f1)': 0.6509, 'answer_relevancy': 0.7686, 'context_entity_recall': 0.5130, 'noise_sensitivity(mode=relevant)': 0.2740},
 'parent_document_retrieval_chain': {'context_recall': 0.7801, 'faithfulness': 0.8026,